In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [5]:
input_csv = "C:/Projects/tickers.csv"
train_csv = "train.csv"
test_csv = "test.csv"

period = "60d" # 8d for 1m,  60d for other ms,  730d for others
interval = "5m" #1m, 2m, 5m, 15m, 30m, 60m, 90m, 1h, 4h, 1d, 5d, 1wk, 1mo, 3mo
prepost = True
random_state = 55

In [6]:
tickers_df = pd.read_csv(input_csv)

ticker_to_company = dict(
    zip(tickers_df["Ticker"], tickers_df["Company"])
)

tickers = tickers_df["Ticker"].dropna().unique().tolist()

In [7]:
train_tickers, test_tickers = train_test_split(
    tickers,
    test_size=0.2,
    random_state=random_state,
    shuffle=True
)

In [8]:
def fetch_ticker_data(ticker):
    try:
        df = yf.Ticker(ticker).history(
            period=period,
            interval=interval,
            prepost=True
        )

        if df.empty:
            return None

        df = df.reset_index()

        df_final = pd.DataFrame({
            "Company": ticker_to_company.get(ticker, "UNKNOWN"),
            "Ticker": ticker,
            "Datetime": df["Datetime"],
            "Open": df["Open"],
            "Close": df["Close"],
            "Ratio": (df["Close"] / df["Open"]) - 1
        })

        return df_final

    except Exception as e:
        print(f"Error ({ticker}): {e}")
        return None

In [9]:
train_data = []

for ticker in train_tickers:
    print(f"📥 Taking : {ticker} for train")
    df = fetch_ticker_data(ticker)
    if df is not None:
        train_data.append(df)

train_df = pd.concat(train_data, ignore_index=True)

📥 Taking : AMD for train
📥 Taking : PYPL for train
📥 Taking : AVGO for train
📥 Taking : ABNB for train
📥 Taking : COST for train
📥 Taking : AMZN for train
📥 Taking : AAPL for train
📥 Taking : TSLA for train
📥 Taking : MSFT for train
📥 Taking : QCOM for train
📥 Taking : BKNG for train
📥 Taking : SBUX for train
📥 Taking : META for train
📥 Taking : ADBE for train
📥 Taking : NFLX for train
📥 Taking : CSCO for train


In [10]:
test_data = []

for ticker in test_tickers:
    print(f"📥 Taking: {ticker} for test")
    df = fetch_ticker_data(ticker)
    if df is not None:
        test_data.append(df)

test_df = pd.concat(test_data, ignore_index=True)

📥 Taking: GOOGL for test
📥 Taking: PEP for test
📥 Taking: NVDA for test
📥 Taking: INTC for test


In [11]:
train_df.to_csv(train_csv, index=False)
test_df.to_csv(test_csv, index=False)

print("✅ İşlem tamamlandı")
print(f"Number of rows for train set: {len(train_df)}")
print(f"Number of rows for test set: {len(test_df)}")

✅ İşlem tamamlandı
Number of rows for train set: 175069
Number of rows for test set: 44901
